# 쿠팡 크롤링 스타터 (Jupyter)

이 노트북은 로컬/사내 Jupyter에서 **Selenium + Chrome**으로 쿠팡 메인 접속까지 테스트하는 최소 예제입니다.
- **Windows/Mac** 모두 가능 (Chrome 설치 필요)
- 크롬드라이버는 `webdriver-manager`가 자동으로 맞춰줍니다
- 실행 순서대로 Shift+Enter 하세요

## 1) 설치
필요 패키지 설치 (한번만 실행)

In [ ]:
!pip install -q selenium webdriver-manager

## 2) 기본 설정 및 드라이버 빌드 함수
- 노트북 환경에서 잘 작동하도록 옵션을 다듬었습니다.
- `headless=True`로 바꾸면 창을 띄우지 않고 실행합니다.

In [ ]:
# -*- coding: utf-8 -*-
import time
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def build_driver(headless: bool = False):
    chrome_opts = Options()
    if headless:
        chrome_opts.add_argument("--headless=new")
    chrome_opts.add_argument("--no-sandbox")
    chrome_opts.add_argument("--disable-dev-shm-usage")
    chrome_opts.add_argument("--window-size=1280,900")
    chrome_opts.add_argument("--lang=ko-KR")
    chrome_opts.add_argument("--disable-blink-features=AutomationControlled")
    chrome_opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_opts.add_experimental_option("useAutomationExtension", False)
    chrome_opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/119.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_opts
    )
    # 간단한 탐지 완화
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"}
    )
    return driver


## 3) 쿠팡 접속 & 스크린샷
- 정상 로딩되면 페이지 제목과 스크린샷 경로를 출력합니다.
- 팝업/쿠키 안내는 있을 수도/없을 수도 있어 예외로 처리합니다.

In [ ]:
driver = build_driver(headless=False)
wait = WebDriverWait(driver, 20)

try:
    url = "https://www.coupang.com/"
    driver.get(url)
    # 타이틀/검색창 등장 대기
    wait.until(lambda d: d.title and len(d.title) > 0)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "input#headerSearchKeyword")))
    print("✅ 접속 성공:", driver.title)
    # 스크린샷 저장
    out = Path("coupang_home.png")
    driver.save_screenshot(str(out))
    print("📸 스크린샷 저장:", out.resolve())
except Exception as e:
    print("❌ 오류:", e)


## 4) (옵션) 검색어 입력 테스트
- 아래 셀의 `keyword`를 바꿔 실행하면, 검색까지 해봅니다.

In [ ]:
try:
    keyword = "동원참치 135g"
    box = driver.find_element(By.CSS_SELECTOR, "input#headerSearchKeyword")
    box.clear()
    box.send_keys(keyword)
    box.submit()  # 엔터와 동일
    # 검색결과 핵심 요소 대기 (상품 리스트 영역 등장)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "ul#productList")))
    print(f"🔎 검색 성공: {keyword}")
    # 확인용 스크린샷
    out = Path("coupang_search.png")
    time.sleep(1.0)
    driver.save_screenshot(str(out))
    print("📸 검색 스크린샷:", out.resolve())
except Exception as e:
    print("(검색 단계) 예외:", e)


## 5) 드라이버 정리
테스트를 마쳤다면 브라우저를 닫아 리소스를 회수하세요.

In [ ]:
try:
    driver.quit()
    print('브라우저 종료')
except: pass

## 문제 해결 가이드
- **Chrome 미설치/버전 불일치**: 로컬에 최신 Chrome 설치. 그래도 안 되면 `!pip show webdriver-manager`로 버전 확인 후 재설치.
- **403/로봇차단**: 반복 실행 간격을 늘리고, 검색/스크롤 사이에 `time.sleep()`을 더 길게 둡니다.
- **팝업/배너 셀렉터 변경**: 개발자도구(F12)로 닫기 버튼의 CSS 선택자를 찾아 코드에 추가하세요.
- **헤드리스 서버**: `build_driver(headless=True)`로 전환. 그래도 차단되면 일반 모드가 나을 수 있습니다.